In data science and statistics, these five tests form the **"Gatekeepers"** of your analysis. If you run a model without checking these first, your results might be mathematically invalid.

Here is the deep dive into when, where, and how to use them.

---

## 1. Normality Test

**The Question:** "Does my data follow a Bell Curve (Normal Distribution)?"

* **When to use:** Before using **Parametric Tests** like t-tests, ANOVA, or Linear Regression. These models assume your residuals/errors are normally distributed.
* **Where to use:** * **Visual:** Histogram or Q-Q Plot (Quantile-Quantile).
* **Statistical:** **Shapiro-Wilk Test** (best for small samples) or **Kolmogorov-Smirnov (K-S) Test** (better for large samples).


* **Decision Rule:** * If $p$-value $> 0.05 \rightarrow$ Data is **Normal**.
* If $p$-value $< 0.05 \rightarrow$ Data is **Not Normal** (use Non-parametric tests like Mann-Whitney).



---

## 2. Homogeneity (Levene’s Test)

**The Question:** "Is the variance (spread) the same across all my groups?"

* **When to use:** Before running an **ANOVA** or an **Independent T-test**. If Group A is very spread out and Group B is very tightly clustered, the comparison isn't "fair."
* **Where to use:** **Levene’s Test** or **Bartlett’s Test**.
* **Decision Rule:** * If $p$-value $> 0.05 \rightarrow$ Variances are equal (**Homoscedasticity**). You are safe to proceed.
* If $p$-value $< 0.05 \rightarrow$ Variances are unequal (**Heteroscedasticity**). You must use "Welch’s T-test" instead.



---

## 3. Reliability (Cronbach’s Alpha)

**The Question:** "Is my measurement tool (like a survey or sensor) consistent?"

* **When to use:** When you have multiple questions measuring the same thing (e.g., 5 questions to measure "Customer Satisfaction").
* **Where to use:** **Cronbach’s Alpha ($\alpha$)**.
* **Decision Rule:** * $\alpha > 0.7 \rightarrow$ **Acceptable** reliability.
* $\alpha > 0.9 \rightarrow$ **Excellent** reliability.
* $\alpha < 0.6 \rightarrow$ Your questions are confusing or measuring different things.



---

## 4. Validity

**The Question:** "Am I actually measuring what I claim to be measuring?"

* **When to use:** During the research design phase and after data collection. Unlike reliability (which is about consistency), validity is about **truth**.
* **Types & Where to use:**
* **Content Validity:** Do experts agree the questions cover the whole topic?
* **Construct Validity:** Does the data correlate with other proven measures? (Use **Factor Analysis** to check this).
* **Criterion Validity:** Does the test predict a real-world outcome? (e.g., Does a high IQ score actually predict job performance?)



---

## 5. Sample Size Computational Test (Power Analysis)

**The Question:** "How many rows of data do I need to be sure my result isn't just luck?"

* **When to use:** **Before** you start collecting data. If your sample is too small, you might miss a real effect (Type II Error). If it's too large, you waste time and money.
* **Where to use:** **G*Power software** or the `statsmodels.stats.power` library in Python.
* **Input Requirements:**
1. **Effect Size:** How big is the difference you expect to see?
2. **Alpha ($\alpha$):** Usually $0.05$ (Significance level).
3. **Power ($1 - \beta$):** Usually $0.80$ (80% chance of finding an effect if it exists).



---

## Summary Cheat Sheet

| Test | Phase | Main Goal | Key Metric |
| --- | --- | --- | --- |
| **Normality** | Pre-Analysis | Check for Bell Curve | Shapiro-Wilk ($p > 0.05$) |
| **Homogeneity** | Pre-Analysis | Check for equal spread | Levene's Test ($p > 0.05$) |
| **Reliability** | Validation | Check for consistency | Cronbach's Alpha ($> 0.7$) |
| **Validity** | Design/Audit | Check for accuracy/truth | Factor Analysis / Expert Review |
| **Sample Size** | Planning | Determine data volume | Power Analysis (Target Power 0.8) |

**Would you like me to show you the Python code to run a Normality and Homogeneity test on a sample dataset?**

To make this practical for your data science portfolio, here is the Python code to run these "Gatekeeper" tests. We will use a sample dataset to check if it is ready for a standard Parametric test (like an ANOVA or T-test).

### 1. The Python Implementation

We will use `scipy.stats` for the mathematical tests and `pingouin` (a very popular library in 2026) for the cleaner output.

```python
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# Create a sample dataset: 2 groups of heights
np.random.seed(42)
group_a = np.random.normal(loc=170, scale=10, size=50) # Normal
group_b = np.random.normal(loc=175, scale=12, size=50) # Normal

# --- 1. NORMALITY TEST (Shapiro-Wilk) ---
# H0: Data is normal | H1: Data is not normal
stat_a, p_a = stats.shapiro(group_a)
stat_b, p_b = stats.shapiro(group_b)

print(f"Group A Normality P-value: {p_a:.4f}")
print(f"Group B Normality P-value: {p_b:.4f}")

# --- 2. HOMOGENEITY TEST (Levene's) ---
# H0: Variances are equal | H1: Variances are not equal
stat_l, p_l = stats.levene(group_a, group_b)
print(f"Levene's Test P-value: {p_l:.4f}")

# --- 3. VISUAL CHECK ---
plt.figure(figsize=(10, 4))
sns.kdeplot(group_a, fill=True, label="Group A")
sns.kdeplot(group_b, fill=True, label="Group B")
plt.title("Visual Check for Normality and Variance")
plt.legend()
plt.show()

```

---

### 2. How to Interpret Your Results

When you run the code above, look specifically for the **p-values**. In these specific "Assumption" tests, **we actually want a high p-value ($> 0.05$)**.

#### **Normality (Shapiro-Wilk)**

* **Result $> 0.05$:** Your data looks like a Bell Curve. You can safely use Linear Regression or T-tests.
* **Result $< 0.05$:** Your data is "Skewed" or has heavy outliers.
* *Solution:* Use a **Log Transformation** or switch to a Non-parametric test (like **Kruskal-Wallis**).



#### **Homogeneity (Levene's)**

* **Result $> 0.05$:** The "spread" of Group A and Group B is similar.
* **Result $< 0.05$:** One group is much wider than the other.
* *Solution:* If you are doing a T-test, use the **Welch's T-test** version (`equal_var=False` in Scipy).



---

### 3. Reliability: Cronbach's Alpha Code

If you are working with survey data or multiple sensors measuring one feature, use this to check **Reliability**:

```python
import pingouin as pg

# Assume 'df' has 3 columns: Q1, Q2, Q3 (all measuring 'Satisfaction')
# alpha_result = pg.cronbach_alpha(data=df[['Q1', 'Q2', 'Q3']])
# print(f"Cronbach's Alpha: {alpha_result[0]}")

```

* **Target:** You want this to be **$> 0.70$**. If it’s lower, your data is "noisy" and inconsistent.

---

### 4. Sample Size: The Power Analysis

Before you start your next project, use this to determine if you have enough data:

```python
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
result = analysis.solve_power(effect_size=0.5, alpha=0.05, power=0.8, ratio=1.0)
print(f"Required Sample Size per Group: {round(result)}")

```

* This tells you the **minimum** number of rows you need to trust your findings.

---

### Summary Table for your Portfolio

| Test Type | Requirement for Parametric Models | If it Fails ($p < 0.05$) |
| --- | --- | --- |
| **Normality** | $p > 0.05$ | Use Mann-Whitney U or Log-Transform |
| **Homogeneity** | $p > 0.05$ | Use Welch's T-test |
| **Reliability** | $\alpha > 0.7$ | Remove inconsistent questions/sensors |

**Would you like me to explain how to perform a "Log Transformation" to fix data that fails the Normality test?**